# Neural processing v6 — derived parameters, one pass, rig-agnostic

**Order:** raw → 1-D temporal Gaussian filter → motion correction → **save shifts** →
PCA *decision on a crop* → PCA (only if kept) → detection → ROIs → traces → ΔF/F → save.

## What changed from v5, and why

**1. Parameters are derived, not typed.** Frame period and µm/pixel come from the rig's own
metadata. Smoothing is declared in **seconds**, the ROI size band in **microns**. Every
parameter that broke a run on 2026-08-09 was one carried between datasets in the wrong
units — a Bruker frame period pasted into a Thorlabs run, a pixel-area band measured at a
different zoom, a correlation threshold tuned against a different noise floor. A pixel area
is not portable. A micron diameter is.

**2. Rig-agnostic metadata.** Reads PrairieView (`TSeries*.xml`) *and* ThorImage
(`Experiment.xml`). The movie file is found by inspection, not by a typed filename.

**3. Shifts are saved the moment they exist**, before anything can overwrite the aligned
stack. Registration is the expensive stage and should never be paid twice.

**4. The PCA decision happens BEFORE PCA runs.** v5 applied PCA in place, destroying the
aligned movie, so the PCA-off control cost a full reload + refilter + reshift. Here the
comparison runs on a crop in seconds and PCA touches the full movie only if it earns it.
Measured on `20260807_M1_Mouse1_Thor`: rank-50 PCA moved the correlation map median from
**0.267 → 0.603** and p95 from **0.373 → 0.849**. It manufactured ~0.34 of correlation at
every pixel in the field, which put a 0.5 threshold below **77.5%** of the FOV and turned
neuropil into ROIs.

**5. Thresholds are read off the data** via a printed sweep, never carried over.

## Unchanged from v5 — same math
The 1-D filter, two-pass rigid registration (`normalization=None`), the streaming
correlation map, the `next_roi` growth rule with its 3×3 seed, trace extraction, ΔF/F.

## Two decisions this notebook asks you to make
1. **PCA on or off** — the crop diagnostic prints the number; keep PCA only if it *increases*
   the separation between cells and background, not merely the correlation everywhere.
2. **`CORR_THRESHOLD`** — from the sweep table, not from another dataset.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **k): return x
from scipy.stats import zscore
from scipy import ndimage
from scipy.ndimage import shift as ndshift, convolve
from skimage.registration import phase_cross_correlation
from skimage.morphology import binary_dilation
import tifffile as tiff
import xml.etree.ElementTree as ET
import json, time, re, gc

## Parameters — the only cell you edit per run

Six knobs. Everything else is derived from the rig metadata in the cells below and printed
so you can check it. `PCA_ENABLE` and `CORR_THRESHOLD` are deliberately `None`: they are
set by the diagnostic cells, from this dataset.

In [ ]:
# ============================ THE ONLY CELL YOU EDIT ============================
DATA_PATH  = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Data to Analyze/'
                  '20260807_M1_Mouse1_Thor/Mouse 1_spontaneous trial/'
                  'Mouse1_spontaneous_trial1_neural')
OUT_PARENT = Path('/grid/courses/data/imagcourse/GECI_Project_Analyzed')
                             # NOT inside the course data tree: 'Data to Analyze' is owned
                             # by whoever uploaded it and is READ-ONLY to us (PermissionError
                             # on mkdir, 2026-08-09). No spaces, so shell/SLURM quoting works.
RUN_NAME   = None            # output subfolder. None = derive from the path. Set a string
                             # for ThorImage runs: their folder names carry no date or mouse
                             # ID, so DATA_PATH.name alone can collide across sessions.
                             # e.g. RUN_NAME = 'thor1_mouse1_spontaneous1'

SOMA_DIAM_UM = (9.0, 20.0)   # biology you are ASSERTING, in MICRONS. Mouse cortical
                             # pyramidal somata ~12-20 um, interneurons ~9-15. Widen the
                             # low end only if you can point at the cells you gain.
GAUSS_SEC    = 0.164         # 1-D temporal smoothing in SECONDS (v4/v5 used 5 frames at
                             # 30.5 Hz = 0.164 s). In seconds so it means the same thing
                             # at any frame rate — 5 FRAMES does not.

PCA_ENABLE     = None        # <- set by the PCA DECISION cell. Leave None.
PCA_RANK       = 50
CORR_THRESHOLD = None        # <- set by the SWEEP cell. Leave None.

DETECT_BIN     = 1           # detection-only temporal bin. 1 is correct whenever the 1-D
                             # filter is on: it has already spent the averaging budget, so
                             # binning on top buys ~8% noise reduction, not 55%.

# ---- overrides: leave None unless the metadata is absent or provably wrong ----
FRAME_PERIOD_OVERRIDE = None
UM_PER_PX_OVERRIDE    = None
MOVIE_FILE_OVERRIDE   = None
# ================================================================================

## Rig metadata — read, never typed

PrairieView and ThorImage record the same physical facts in completely different schemas.
This is the cell that makes one notebook work on both rigs, and it is the cell that would
have caught the 2026-08-09 failure: a Thorlabs run inherited a Bruker frame period by hand.

Two traps handled explicitly:
- PrairieView's `micronsPerPixel` carries an **X, a Y and a Z** entry. Only X and Y are
  in-plane; averaging Z in silently corrupts the scale.
- ThorImage reports `frameRate` alongside `averageMode`/`averageNum`. If frame averaging is
  enabled, the effective rate is **not** `frameRate` — the cell says so rather than guessing.

In [ ]:
def read_metadata(folder):
    """Rig-agnostic. -> dict(frame_period_s, um_per_px, n_frames, source, xml, notes)."""
    folder = Path(folder)
    md = {"source": None, "xml": None, "notes": [], "n_frames": None, "fov_um": None}

    thor = folder / "Experiment.xml"
    pv   = sorted([p for p in folder.glob("TSeries*.xml") if "Voltage" not in p.name],
                  key=lambda p: len(p.name))

    if thor.exists():                                    # ---------------- ThorImage
        root = ET.parse(str(thor)).getroot()
        md["source"], md["xml"] = "ThorImage", thor
        lsm = root.find(".//LSM")
        if lsm is None:
            raise RuntimeError(f"{thor}: no <LSM> block")
        fr = float(lsm.get("frameRate"))
        md["frame_period_s"] = 1.0 / fr

        px, w, nx = lsm.get("pixelWidthUM"), lsm.get("widthUM"), lsm.get("pixelX")
        md["um_per_px"] = float(px) if px else float(w) / float(nx)
        if px and w and nx:                              # two routes must agree
            alt = float(w) / float(nx)
            if abs(alt - float(px)) > 0.02 * alt:
                md["notes"].append(f"pixelWidthUM={px} disagrees with widthUM/pixelX="
                                   f"{alt:.4f}; using {md['um_per_px']:.4f}")
        md["fov_um"] = float(w) if w else None

        am, an = lsm.get("averageMode", "0"), int(lsm.get("averageNum", "1") or 1)
        if am not in ("0", None, "") and an > 1:
            md["notes"].append(f"FRAME AVERAGING ON (mode {am}, n {an}) -> effective rate "
                               f"may be {fr/an:.4f} Hz, not {fr}. CONFIRM before trusting "
                               f"any timing.")
        st, tl = root.find(".//Streaming"), root.find(".//Timelapse")
        n = (st.get("frames") if st is not None else None) or \
            (tl.get("timepoints") if tl is not None else None)
        md["n_frames"] = int(n) if n else None

    elif pv:                                             # ---------------- PrairieView
        x = pv[0]
        root = ET.parse(str(x)).getroot()
        md["source"], md["xml"] = "PrairieView", x
        fr_el = root.findall(".//Frame")
        t = np.array([float(f.get("relativeTime")) for f in fr_el
                      if f.get("relativeTime") is not None])
        if len(t) < 2:
            raise RuntimeError(f"{x}: fewer than two frame times")
        md["frame_period_s"] = float(np.median(np.diff(t)))   # median: robust to jitter
        md["n_frames"] = len(fr_el)
        umpp = {}
        for sv in root.findall(".//PVStateValue"):
            if sv.get("key") == "micronsPerPixel":
                for iv in sv.iter():
                    if iv.get("index") in ("XAxis", "YAxis") and iv.get("value"):
                        umpp[iv.get("index")] = float(iv.get("value"))   # ZAxis EXCLUDED
        if not umpp:
            raise RuntimeError(f"{x}: micronsPerPixel absent")
        if len(umpp) == 2 and abs(umpp["XAxis"] - umpp["YAxis"]) > 0.02 * max(umpp.values()):
            md["notes"].append(f"non-square pixels {umpp} — using the mean; ROI areas are "
                               f"approximate")
        md["um_per_px"] = float(np.mean(list(umpp.values())))
    else:
        raise RuntimeError(f"no Experiment.xml (ThorImage) or TSeries*.xml (PrairieView) "
                           f"in {folder}")
    return md


def find_movie(folder):
    """The imaging stack, by inspection. OME series -> first file of the single channel.
    Otherwise the LARGEST tif: ThorImage writes previews and thumbnails alongside it."""
    folder = Path(folder)
    tifs = [p for p in folder.iterdir()
            if p.suffix.lower() in (".tif", ".tiff") and not p.name.startswith(".")]
    if not tifs:
        raise RuntimeError(f"no tif files in {folder}")
    ome = sorted([p for p in tifs if p.name.lower().endswith((".ome.tif", ".ome.tiff"))])
    if ome:
        ch = sorted({m.group(1) for p in ome for m in [re.search(r"_Ch(\d+)_", p.name)] if m})
        if len(ch) > 1:
            raise RuntimeError(f"multiple channels {ch} in {folder} — expected one")
        return ome[0], len(ome)
    return max(tifs, key=lambda p: p.stat().st_size), 1

In [ ]:
META = read_metadata(DATA_PATH)
MOVIE_PATH, _n_files = ((Path(MOVIE_FILE_OVERRIDE), 1) if MOVIE_FILE_OVERRIDE
                        else find_movie(DATA_PATH))

FRAME_PERIOD = FRAME_PERIOD_OVERRIDE or META["frame_period_s"]
UM_PER_PX    = UM_PER_PX_OVERRIDE    or META["um_per_px"]
frame_rate   = 1.0 / FRAME_PERIOD

GAUSS_WIDTH   = max(1, int(round(GAUSS_SEC / FRAME_PERIOD)))
_d_px         = np.array(SOMA_DIAM_UM) / UM_PER_PX
SIZE_MIN, SIZE_MAX = (int(round(np.pi / 4 * _d_px[0] ** 2)),
                      int(round(np.pi / 4 * _d_px[1] ** 2)))
SIZE_GROW_CAP = 2 * SIZE_MAX      # MUST exceed SIZE_MAX, or every object stops AT the cap,
                                  # passes the size test, and the filter stops filtering
MAX_ROIS_CAP   = 2000
BORDER_PX_MIN  = 5
DFF_PERCENTILE = 15
BLEACH_WARN_PCT = 20

RUN_NAME = RUN_NAME or DATA_PATH.name
out      = OUT_PARENT / RUN_NAME          # every later cell writes here

print(f"metadata     : {META['source']}  ({Path(META['xml']).name})")
print(f"movie        : {MOVIE_PATH.name}  "
      f"({MOVIE_PATH.stat().st_size/1e9:.1f} GB"
      + (f", {_n_files} files)" if _n_files > 1 else ")"))
print(f"frame period : {FRAME_PERIOD:.6f} s  ({frame_rate:.3f} Hz)"
      + ("   [OVERRIDE]" if FRAME_PERIOD_OVERRIDE else ""))
print(f"n_frames(md) : {META['n_frames']}")
print(f"scale        : {UM_PER_PX:.4f} um/px"
      + ("   [OVERRIDE]" if UM_PER_PX_OVERRIDE else "")
      + (f"   FOV {META['fov_um']:.0f} um" if META["fov_um"] else ""))
print(f"smoothing    : {GAUSS_SEC} s  ->  GAUSS_WIDTH {GAUSS_WIDTH} frames")
print(f"soma band    : {SOMA_DIAM_UM[0]}-{SOMA_DIAM_UM[1]} um  =  "
      f"{_d_px[0]:.1f}-{_d_px[1]:.1f} px across  =  AREA {SIZE_MIN}-{SIZE_MAX} px"
      f"   (grow cap {SIZE_GROW_CAP})")
for _n in META["notes"]:
    print(f"  ** NOTE: {_n}")
if GAUSS_WIDTH < 2:
    print("  ** WARNING: GAUSS_WIDTH < 2 frames — smoothing is effectively off at this rate.")
if _d_px[0] < 5:
    print(f"  ** WARNING: a {SOMA_DIAM_UM[0]} um soma is only {_d_px[0]:.1f} px across here; "
          f"ROI areas below ~5 px diameter are unreliable.")
print(f"output       : {out}")

## Optional — Bruker multi-file series concatenation

**You normally do not need this.** tifffile follows the OME-XML `TiffData` references, so
reading the *first* `.ome.tif` of a PrairieView series returns **every frame across every
file**. `find_movie()` + the load cell already handle it.

**The trap this cell exists to avoid:** reading each file with a plain `tiff.imread()` and
concatenating gives you the whole series **once per file** — a 3-file run yields 3× the
frames, every one duplicated, with no error. `is_ome=False` is what makes a per-file read
actually mean one file.

So this cell checks whether aggregation works *before* doing anything, and concatenates only
if it genuinely failed (aborted acquisition, damaged OME header, renamed files). The result
is written to the **output** tree, never into the course-managed raw data, and
`MOVIE_PATH` is repointed at it.

In [ ]:
# ---- OPTIONAL: single-file rebuild for a PrairieView multi-file series ----------
CONCAT_IF_NEEDED = False        # True only if the check below reports a mismatch

if CONCAT_IF_NEEDED and META["source"] == "PrairieView":
    _nat = lambda p: [int(t) if t.isdigit() else t.lower()
                      for t in re.split(r"(\d+)", p.name)]        # 10 sorts after 2
    _series = sorted([p for p in DATA_PATH.glob("*.ome.tif")
                      if not p.name.startswith(".")], key=_nat)
    if not _series:
        raise RuntimeError(f"no .ome.tif files in {DATA_PATH}")
    _ch = sorted({m.group(1) for p in _series
                  for m in [re.search(r"_Ch(\d+)_", p.name)] if m})
    if len(_ch) > 1:
        raise RuntimeError(f"multiple channels {_ch} present — restrict the glob first")

    with tiff.TiffFile(str(_series[0])) as _tf:                   # what does file 1 claim?
        _agg = _tf.series[0].shape[0] if _tf.series else len(_tf.pages)
    print(f"{len(_series)} file(s), channel {_ch or '?'} | XML frames "
          f"{META['n_frames']} | first file's OME series reports {_agg}")

    if META["n_frames"] and _agg == META["n_frames"]:
        print("  OME aggregation is CORRECT — concatenation not needed, skipping.")
    else:
        _dst_dir = OUT_PARENT / DATA_PATH.name        # never write into the raw tree
        _dst_dir.mkdir(parents=True, exist_ok=True)
        _dst = _dst_dir / "concatenated.tif"

        if _dst.exists():
            print(f"  reusing existing {_dst}")
        else:
            _meta = []
            for _f in _series:                        # page counts WITHOUT loading pixels
                with tiff.TiffFile(str(_f)) as _tf:
                    _meta.append((len(_tf.pages), _tf.pages[0].shape, _tf.pages[0].dtype))
            if len({m[1] for m in _meta}) != 1:
                raise RuntimeError(f"frame shapes differ across files: {_meta}")
            _tot = sum(m[0] for m in _meta)
            if META["n_frames"] and _tot != META["n_frames"]:
                print(f"  ** WARNING: {_tot} frames on disk vs {META['n_frames']} in the "
                      f"XML — partial upload or aborted run.")
            print(f"  concatenating {_tot} frames -> {_dst}")

            # ONE allocation, filled in place. The list-then-concatenate idiom holds
            # every file AND the joined copy at once: 2x the movie at peak.
            _buf, _k = np.empty((_tot, *_meta[0][1]), _meta[0][2]), 0
            for _f in _series:
                _a = tiff.imread(str(_f), is_ome=False)   # <- load-bearing: one file only
                if _a.ndim == 2:
                    _a = _a[None]
                _buf[_k:_k + len(_a)] = _a
                _k += len(_a)
                del _a
            assert _k == _tot, f"filled {_k} of {_tot} frames"
            tiff.imwrite(_dst, _buf, bigtiff=True)
            del _buf
            gc.collect()

        MOVIE_FILE_OVERRIDE = _dst
        MOVIE_PATH = _dst
        print(f"  MOVIE_PATH -> {MOVIE_PATH}")

## Load
`frames` must be (T, Y, X). The frame count is checked against the metadata — a partial
upload silently truncates and poisons everything downstream.

In [ ]:
t0 = time.time()
frames = tiff.imread(str(MOVIE_PATH))
frames = np.squeeze(frames)
if frames.ndim == 2:
    frames = frames[None]
assert frames.ndim == 3, (f"expected (T, Y, X), got {frames.shape} — multi-channel or "
                          f"multi-plane data needs an explicit axis choice HERE")
T, Y, X = frames.shape
time_vector = np.arange(T) * FRAME_PERIOD
print(f"movie {frames.shape} {frames.dtype}  ({T*FRAME_PERIOD:.0f}s @ {frame_rate:.2f} Hz, "
      f"{frames.nbytes/1e9:.1f} GB raw, load {time.time()-t0:.0f}s)")
if META["n_frames"] and META["n_frames"] != T:
    print(f"  ** WARNING: metadata reports {META['n_frames']} frames, loaded {T}. "
          f"Partial upload or aborted acquisition — resolve before batching this run.")

In [ ]:
original_anatomy = frames.mean(0)
plt.figure(figsize=(4, 4))
plt.imshow(original_anatomy, cmap="gray",
           vmin=np.percentile(original_anatomy, 1), vmax=np.percentile(original_anatomy, 99))
plt.title("original anatomy (raw mean)"); plt.axis("off"); plt.show()

## Step 1 — 1-D temporal Gaussian filter (instructor code, VERBATIM)
Each pixel's time series is convolved with a 1-D Gaussian (σ = `GAUSS_WIDTH` frames).
Per-frame shot noise averages away; every frame becomes a local time-average with real
structure for the registration step. Fidelity notes: kernel and convolution are the
instructor notebook's exact code (only `gaussWidth` now reads from the parameters
cell); `ndimage.convolve` keeps uint16 (rounding ≤0.5 count — negligible at these
intensities); the 20-tap kernel's one-sample asymmetry is retained (see header).
The raw movie is deleted afterwards — everything downstream uses the filtered movie.

In [ ]:
gaussWidth = GAUSS_WIDTH
filterSize = 2*gaussWidth
pix4filt   = np.arange(-filterSize,filterSize)**2

# Temporal filters operate in 1D, so we need a 1D Gaussian bump:
gFilt1D = np.exp(-(pix4filt)/(2*(gaussWidth)**2))
gFilt1D = gFilt1D/gFilt1D.sum()
gFilt1D = np.expand_dims(np.expand_dims(gFilt1D,axis=1),axis=2)
clearFrames1D = ndimage.convolve(frames,gFilt1D)
clearFrames1D = np.array(clearFrames1D)

del frames          # raw no longer needed; frees ~0.5x movie of RAM
print(f"filtered movie: {clearFrames1D.shape} {clearFrames1D.dtype} "
      f"(sigma {gaussWidth} frames = {gaussWidth*FRAME_PERIOD*1e3:.0f} ms)")

## Step 2 — motion correction on the FILTERED movie (two-pass rigid, v2 algorithm)
Identical two-pass structure and shift interpolation to v2, now consuming
`clearFrames1D` per the restructured ordering. **One flagged parameter deviation**
(see the comment in the next cell): `normalization=None` in the phase-correlation
call — measured 14× lower registration error on temporally-filtered frames than the
skimage default; with the default, this ordering silently loses ~1 px of accuracy.
Engineering retained from v3 (no math changes): float32 everywhere; the pass-1
aligned stack is never materialized — only its **mean** is accumulated to build the
refined template. Peak memory ≈ filtered uint16 + ONE aligned float32 stack ≈
1.5× movie-float32 (~43 GB at 512²×27k).

In [ ]:
# FLAGGED DEVIATION from v2 defaults, required by the v4 ordering (measured, not
# assumed): temporal filtering removes temporal noise but leaves per-frame SPATIAL
# high-frequency noise, and skimage's default normalization="phase" whitens the
# spectrum — upweighting exactly that noise. On ground-truth synthetic data the
# default gave 1.23 px RMS registration error on filtered frames; classic
# cross-correlation (normalization=None) gave 0.086 px (r=0.999 vs truth).
MC_NORMALIZATION = None      # <-- REQUIRED for the v4 ordering. Do not change to "phase".

def estimate_shifts(mov, template, upsample=10):
    ys, xs = np.empty(len(mov)), np.empty(len(mov))
    kw = {"upsample_factor": upsample}
    try:
        phase_cross_correlation(template, mov[0], normalization=MC_NORMALIZATION, **kw)
        kw["normalization"] = MC_NORMALIZATION
    except TypeError:              # very old skimage: kwarg absent; default applies
        print("WARNING: this skimage has no `normalization` kwarg — registration will "
              "use the phase-normalized default, which measured ~14x worse on filtered "
              "frames. Upgrade skimage if shifts look noisy.")
    for i in tqdm(range(len(mov)), desc="shifts", leave=False):
        (dy, dx), _, _ = phase_cross_correlation(template, mov[i], **kw)
        ys[i], xs[i] = dy, dx
    return ys, xs

# pass 1: estimate against the filtered-movie mean, accumulate the aligned MEAN only
filtered_anatomy = clearFrames1D.mean(0)
y1, x1 = estimate_shifts(clearFrames1D, filtered_anatomy)
acc = np.zeros((Y, X), np.float64)
for i in tqdm(range(T), desc="template", leave=False):
    acc += ndshift(clearFrames1D[i].astype(np.float32), (y1[i], x1[i]))
aligned_anatomy = (acc / T).astype(np.float32)

# pass 2: re-estimate the UNSHIFTED filtered frames against the refined template
# (no double interpolation — same structure as v2)
y2, x2 = estimate_shifts(clearFrames1D, aligned_anatomy)
total_shift_1 = np.hypot(y1, x1)
total_shift_2 = np.hypot(y2, x2)

In [ ]:
# apply pass-2 shifts -> the ONE aligned float32 stack used everywhere downstream
final_frames = np.empty((T, Y, X), np.float32)
for i in tqdm(range(T), desc="apply", leave=False):
    final_frames[i] = ndshift(clearFrames1D[i].astype(np.float32), (y2[i], x2[i]))
final_anatomy = final_frames.mean(0)
max_shift = float(np.abs(np.concatenate([y2, x2])).max())
del clearFrames1D      # aligned stack replaces it; frees ~0.5x movie
print(f"max |shift| = {max_shift:.2f} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(time_vector, total_shift_1, lw=.6, label="pass 1")
axes[0].plot(time_vector, total_shift_2, lw=.6, label="pass 2")
axes[0].set(xlabel="time (s)", ylabel="total shift (px)"); axes[0].legend()
for ax, img, name in ((axes[1], original_anatomy, "before"), (axes[2], final_anatomy, "after")):
    ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

## Save the shifts — immediately

Registration is the expensive stage: two `phase_cross_correlation` passes over every frame.
Written here, the moment they exist, **before** PCA or anything else can overwrite the
aligned stack. With these on disk any later re-run — new threshold, new size band, PCA
toggled — costs load + filter + one shift pass, never re-estimation.

This cell is the direct lesson of 2026-08-09, when an in-place PCA destroyed the aligned
movie and the PCA-off control cost a full reload.

In [ ]:
import os
if not OUT_PARENT.parent.is_dir():          # never fabricate a mistyped root
    raise FileNotFoundError(f"parent of OUT_PARENT does not exist: {OUT_PARENT.parent}")
if not os.access(OUT_PARENT.parent, os.W_OK):
    raise PermissionError(f"no write access under {OUT_PARENT.parent}. The course data tree "
                          f"is read-only; use a root you own, e.g. {Path.home() / 'ca_output'}")
OUT_PARENT.mkdir(exist_ok=True)             # parents=False: one level only
out.mkdir(exist_ok=True)

np.save(out / "shifts_yx.npy", np.stack([y2, x2]))
np.save(out / "mean_img_prePCA.npy", final_anatomy)
print(f"shifts -> {out/'shifts_yx.npy'}   max|shift| "
      f"{np.abs(np.stack([y2, x2])).max():.2f} px")
print(f"outputs will go to: {out}")

## PCA decision — made BEFORE PCA touches the full movie

PCA reconstruction projects every pixel onto a shared rank-`PCA_RANK` basis, which raises
pairwise correlations **by construction**. That is not automatically a gain: if it lifts
background and cells by the same amount, the correlation map gets brighter while telling you
*less*, and the threshold ends up below the neuropil.

So the test is not "is the map higher" but **"is the gap between cells and background
wider"**. The cell below computes both maps on a centre crop — seconds, not minutes — and
prints the separation each way.

Measured on `20260807_M1_Mouse1_Thor`: PCA raised the median from 0.267 to 0.603 while the
cell-vs-background gap did *not* widen proportionally. 128 ROIs, almost all neuropil.

In [ ]:
def pca_denoise_inplace(mov_f32, rank, chunk=2000):
    """Rank-`rank` reconstruction of (T,Y,X) float32 movie, written back IN PLACE.
    Memory-safe: the (T, Y*X) matrix is a reshape VIEW of the movie (no copy);
    only the factors + one row-block are allocated."""
    Tn, Yn, Xn = mov_f32.shape
    M = mov_f32.reshape(Tn, Yn * Xn)          # view, not a copy
    t0 = time.time()
    try:
        from sklearn.utils.extmath import randomized_svd
        U, S, Vt = randomized_svd(M, n_components=rank, random_state=0)
    except ImportError:
        from scipy.sparse.linalg import svds
        U, S, Vt = svds(M, k=rank)
    for i in range(0, Tn, chunk):
        M[i:i+chunk] = (U[i:i+chunk] * S) @ Vt
    print(f"PCA denoise rank {rank}: {time.time()-t0:.0f}s")

In [ ]:
def bin_movie(mov, n):
    if n <= 1:
        return mov
    Tb = (len(mov) // n) * n
    return mov[:Tb].reshape(-1, n, *mov.shape[1:]).mean(axis=1)


def neighbor_corr_map(mov):
    """Pearson r of each pixel vs the SUM of its 8 neighbours — exact, one streaming pass,
    no movie copies (proven equal to the per-pixel pearsonr loop to 1e-6)."""
    Tn = len(mov)
    k  = np.ones((3, 3))
    sx = np.zeros(mov.shape[1:]); sxx = np.zeros(mov.shape[1:])
    ss = np.zeros(mov.shape[1:]); sss = np.zeros(mov.shape[1:]); sxs = np.zeros(mov.shape[1:])
    for t in range(Tn):
        f = mov[t].astype(np.float64)
        s = convolve(f, k, mode="constant") - f
        sx += f; sxx += f * f; ss += s; sss += s * s; sxs += f * s
    num = Tn * sxs - sx * ss
    den = np.sqrt((Tn * sxx - sx**2) * (Tn * sss - ss**2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, 0.0)      # constant pixels -> 0, never NaN


def fast_pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0

def next_roi(corr_map, mov, corr_threshold, grow_cap):
    i, j = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    # SEED = 3x3 BLOCK, not 1 px. The correlation map scores a pixel against the SUM of its
    # 8 neighbours (~sqrt(8) SNR advantage), but growth then compared that same pixel against
    # ONE RAW neighbour - a far noisier quantity, so the two use incompatible scales. Seeds
    # whose map value was 0.5-0.67 could never clear the same threshold pixel-to-pixel and
    # died at 1 px: 2780 of 3885 attempts on the gold run, while the cells they sat on stayed
    # unclaimed at corr 0.86. Seeding with the 3x3 SUM makes the first comparison match the
    # statistic the map already validated. (Still a COPY, not a view - fix #2 above.)
    this_roi = np.zeros(corr_map.shape, np.uint8)
    this_roi[max(i - 1, 0):i + 2, max(j - 1, 0):j + 2] = 1
    ys, xs = np.where(this_roi)
    this_trace = mov[:, ys, xs].sum(1).astype(np.float64)
    used = corr_map.copy(); used[this_roi.astype(bool)] = 0
    growing = True
    while growing and this_roi.sum() < grow_cap:
        growing = False
        ring = np.argwhere(binary_dilation(this_roi, np.ones((3, 3))) ^ this_roi.astype(bool))
        add = np.zeros(len(this_trace))
        for (yy, xx) in ring:
            if used[yy, xx] != 0:
                px = mov[:, yy, xx].astype(np.float64)
                if fast_pearson(this_trace, px) > corr_threshold:
                    growing = True
                    this_roi[yy, xx] = 1
                    used[yy, xx] = 0
                    add += px
        this_trace += add
    return this_roi, this_roi.sum(), used


In [ ]:
# ---- PCA on vs off, on a centre crop ----
CROP = 128
cy, cx = Y // 2, X // 2
_ys = slice(max(0, cy - CROP // 2), cy + CROP // 2)
_xs = slice(max(0, cx - CROP // 2), cx + CROP // 2)

crop_off = np.ascontiguousarray(final_frames[:, _ys, _xs])
crop_on  = crop_off.copy()
pca_denoise_inplace(crop_on, min(PCA_RANK, len(crop_on) - 1))

_an = final_anatomy[_ys, _xs][2:-2, 2:-2]
_hi, _dim = _an > np.percentile(_an, 99), _an < np.percentile(_an, 50)

print(f"{'':10}{'median':>9}{'p90':>9}{'p95':>9}{'max':>9}{'bright':>9}{'dim':>9}{'GAP':>9}")
_res = {}
for _nm, _mv in (("PCA OFF", crop_off), ("PCA ON", crop_on)):
    _cm = neighbor_corr_map(bin_movie(_mv, DETECT_BIN))[2:-2, 2:-2]
    _gap = np.median(_cm[_hi]) - np.median(_cm[_dim])
    _res[_nm] = (_cm, _gap)
    print(f"{_nm:10}{np.median(_cm):9.3f}{np.percentile(_cm,90):9.3f}"
          f"{np.percentile(_cm,95):9.3f}{_cm.max():9.3f}"
          f"{np.median(_cm[_hi]):9.3f}{np.median(_cm[_dim]):9.3f}{_gap:9.3f}")

_g_off, _g_on = _res["PCA OFF"][1], _res["PCA ON"][1]
print(f"\nseparation (bright - dim):  OFF {_g_off:+.3f}   ON {_g_on:+.3f}   "
      f"change {_g_on - _g_off:+.3f}")
print("KEEP PCA only if the GAP grows. A higher map with the same or smaller gap is\n"
      "manufactured correlation: it lifts neuropil and somata together and drags the\n"
      "usable threshold down into the background.")
print(f"\n->  suggested PCA_ENABLE = {_g_on > _g_off + 0.02}"
      f"   (set it in the parameters cell and re-run that cell)")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(_an, cmap="gray"); ax[0].set_title("anatomy (crop)")
for a, nm in zip(ax[1:], ("PCA OFF", "PCA ON")):
    a.imshow(_res[nm][0], cmap="gray", vmin=0, vmax=1)
    a.set_title(f"{nm}  gap {_res[nm][1]:+.3f}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

del crop_on, crop_off; gc.collect()

## Commit to the PCA decision

`PCA_ENABLE` must now be `True` or `False` in the parameters cell — not `None`. If you keep
PCA, this cell writes into `final_frames` **in place** and the pre-PCA movie is gone; the
shifts saved above are what make that recoverable cheaply.

In [ ]:
if PCA_ENABLE is None:
    raise RuntimeError("PCA_ENABLE is still None — run the PCA decision cell, set it in the "
                       "parameters cell, re-run that cell, then come back here.")
if PCA_ENABLE:
    pca_denoise_inplace(final_frames, min(PCA_RANK, T - 1))
    print(f"PCA applied in place: rank {min(PCA_RANK, T-1)}")
else:
    print("PCA skipped — final_frames is the aligned, filtered movie.")

In [ ]:
DETECT_BIN_N = (max(1, int(round(frame_rate / 6.0))) if DETECT_BIN == "auto"
                else int(DETECT_BIN))
detect_mov = bin_movie(final_frames, DETECT_BIN_N)
print(f"detection movie: bin x{DETECT_BIN_N} -> {len(detect_mov)} frames @ "
      f"{frame_rate/DETECT_BIN_N:.1f} Hz")
t0 = time.time()
correlation_map = neighbor_corr_map(detect_mov)
print(f"correlation map computed in {time.time()-t0:.0f}s")

## Border guard + NaN hygiene (directives 5 & 6)
Shifting fills edges with zeros in *some frames only* — edge-pixel traces become gated
by the shift time course, which is **shared across all edge pixels**, so they correlate
near 1.0 with each other and grow fake "cells" of pure motion artifact. The dead band
is exactly the maximum shift, so the border is `max(BORDER_PX_MIN, ceil(max|shift|)+1)`.
NaNs (none can arise from the streaming map, but belt-and-suspenders for any edit) would
otherwise win `argmax` and hijack ROI seeding.

In [ ]:
border = max(BORDER_PX_MIN, int(np.ceil(max_shift)) + 1)
corr_bordered = np.nan_to_num(correlation_map, nan=0.0, posinf=0.0, neginf=0.0).copy()
corr_bordered[:border, :] = 0; corr_bordered[-border:, :] = 0
corr_bordered[:, :border] = 0; corr_bordered[:, -border:] = 0
print(f"border zeroed: {border} px (max shift {max_shift:.2f})")

# ---- THRESHOLD RECALIBRATION (required by the v4 ordering) ----
# The 1-D temporal filter raises correlations EVERYWHERE, background included, so a
# threshold tuned on unfiltered data over-detects. On ground-truth synthetic data with
# this ordering: background median 0.28 / p95 0.75, true cells 0.60-0.85; threshold 0.3
# gave 16 ROIs and missed a cell, 0.5-0.6 gave 7-8 ROIs and found all three.
# RULE OF THUMB: set CORR_THRESHOLD near the map's 95th percentile, then eyeball the
# ROI overlay. The printout below gives you that number for THIS run.
_vals = corr_bordered[corr_bordered > 0]
print(f"corr map: median {np.median(_vals):.3f}  p95 {np.percentile(_vals, 95):.3f}  "
      f"max {corr_bordered.max():.3f}   <- CORR_THRESHOLD is {CORR_THRESHOLD}")
if CORR_THRESHOLD < np.percentile(_vals, 90):
    print(f"  WARNING: threshold is below the map's 90th percentile "
          f"({np.percentile(_vals, 90):.3f}) — expect over-detection of background.")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
axes[0].set_title("anatomy"); axes[0].axis("off")
im = axes[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"correlation map (max {corr_bordered.max():.2f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()
if corr_bordered.max() < CORR_THRESHOLD:
    print(f"WARNING: corr map max {corr_bordered.max():.2f} < threshold {CORR_THRESHOLD}"
          f" — no ROIs will grow. Increase DETECT_BIN (see sweep) before proceeding.")

## Threshold sweep — read `CORR_THRESHOLD` off this table

A correlation threshold is meaningless as an absolute number: it depends on SNR, frame rate,
filtering and whether PCA ran. Carrying one between datasets is what produced a 128-ROI
neuropil field on one run and 0 ROIs on the next.

The sweep runs the real growth rule at candidate thresholds taken from **this** map's own
percentiles. Read it as:

- **accepted** rises then falls — take the high side of the plateau; a higher threshold
  buys boundary-stopping, which is what keeps adjacent cells from merging.
- **medarea** should sit inside `SIZE_MIN`–`SIZE_MAX`. Climbing well above it means growth
  is bridging into neuropil.
- **atcap** above a handful means runaway growth: threshold too low.
- **A large pile below `SIZE_MIN`** in the size distribution means the size FLOOR is
  rejecting real cells and the threshold is not your problem.

In [ ]:
_v = corr_bordered[corr_bordered > 0]
print("corr map: " + "  ".join(f"p{k}={np.percentile(_v,k):.3f}"
      for k in (50, 75, 90, 95, 97, 99)) + f"  max={_v.max():.3f}")

# Where the threshold BELONGS: above the background's upper tail, below most somatic
# pixels. These two numbers place it; the percentiles alone do not, because what
# fraction of a FOV is cells varies with density.
_hi  = final_anatomy > np.percentile(final_anatomy, 99)     # somata
_dim = final_anatomy < np.percentile(final_anatomy, 50)     # background
_lo_bound, _hi_bound = np.percentile(corr_bordered[_dim], 99), np.percentile(corr_bordered[_hi], 25)
print(f"corr @ BRIGHT: med {np.median(corr_bordered[_hi]):.3f}  p25 {_hi_bound:.3f}")
print(f"corr @ DIM   : med {np.median(corr_bordered[_dim]):.3f}  p99 {_lo_bound:.3f}")
if _lo_bound < _hi_bound:
    print(f"-> usable window {_lo_bound:.3f} - {_hi_bound:.3f}, "
          f"suggested CORR_THRESHOLD = {max(_lo_bound, _hi_bound):.3f}")
else:
    print("-> ** the two populations OVERLAP: no single threshold separates cells from\n"
          "      background. The problem is upstream (PCA on? residual motion?), not here.")
print()

_BINS = [(9, 19), (20, 49), (50, 79), (80, 149), (150, 249), (250, 399), (400, 10**9)]
print(f"{'thr':>5}{'attempts':>10}{'accept':>8}{'medarea':>9}{'atcap':>7}   size distribution")
for _thr in np.round(np.percentile(_v, [80, 85, 90, 93, 95, 97]), 3):
    _s, _um = [], corr_bordered.copy()
    while _um.max() > _thr and len(_s) < 20000:
        _roi, _sz, _um = next_roi(_um, detect_mov, _thr, SIZE_GROW_CAP)
        _s.append(int(_sz))
    _s = np.array(_s)
    _acc = _s[(_s > SIZE_MIN) & (_s < SIZE_MAX)]
    print(f"{_thr:5.2f}{len(_s):10d}{len(_acc):8d}"
          f"{(np.median(_acc) if len(_acc) else 0):9.0f}"
          f"{int((_s >= SIZE_GROW_CAP).sum()):7d}   "
          + " ".join(f"{lo}-{'inf' if hi > 10**8 else hi}:"
                     f"{int(((_s >= lo) & (_s <= hi)).sum())}" for lo, hi in _BINS)
          + ("   <- TRUNCATED" if len(_s) >= 20000 else ""))
print("\nSet CORR_THRESHOLD in the parameters cell, re-run it, then run extraction below.")

## ROI extraction (directive 7) — data-driven, indexing-safe
Changes vs v2, each load-bearing:
1. **No `n_rois` hardcode.** Extraction runs until the correlation map is *exhausted*
   (no remaining seed above `CORR_THRESHOLD`) — the data decides the count. A cap of
   `MAX_ROIS_CAP` exists purely as a runaway guard (each attempt consumes ≥1 seed
   pixel, so termination is guaranteed regardless).
2. **`this_trace = ....copy()`** — in v2 this was a VIEW into the movie, and `+=`
   silently wrote the growing ROI sum back into `final_frames`, corrupting every later
   ROI's correlations. This was the most dangerous bug in the notebook.
3. **Labels == rows.** v2 stamped `roi_map` with the attempt index but compacted the
   trace array — map label k did not index trace row k. Here masks are collected and
   labeled 1..n in exactly trace-row order.
4. Growth runs on the detection movie (`DETECT_BIN`; with the v4 ordering that is
   normally the filtered movie itself, bin=1); traces come from the **full-rate**
   movie in the next cell.
5. `pearsonr` replaced by the identical dot-product formula (scale-invariant Pearson,
   equal to scipy to 1e-10) — turns hours into minutes at full FOV.

**Check the ROI overlay against `CORR_THRESHOLD` before trusting the count** — with the
filtered ordering the threshold is the single most sensitive parameter (see the
recalibration printout above).

In [ ]:
masks, sizes_all = [], []
used_map = corr_bordered.copy()
t0 = time.time()
while used_map.max() > CORR_THRESHOLD and len(masks) < MAX_ROIS_CAP:
    roi, size, used_map = next_roi(used_map, detect_mov, CORR_THRESHOLD, SIZE_GROW_CAP)
    sizes_all.append(int(size))
    if SIZE_MIN < size < SIZE_MAX:
        masks.append(roi.astype(bool))
n_rej = len(sizes_all) - len(masks)
print(f"{len(masks)} ROIs accepted, {n_rej} size-rejected "
      f"(attempt sizes min/med/max {min(sizes_all)}/{int(np.median(sizes_all))}/"
      f"{max(sizes_all)}) in {time.time()-t0:.0f}s")

roi_map = np.zeros((Y, X), np.uint16)
for k, m in enumerate(masks):
    roi_map[m] = k + 1        # label k+1 == trace row k, ALWAYS  (fix #3)

In [ ]:
# ---- did the extraction work? the three numbers that decide it ----
_areas = np.array([m.sum() for m in masks]) if masks else np.array([0])
_s     = np.array(sizes_all)
print(f"accepted {len(masks)} | attempts {len(_s)} | "
      f"too-small {int((_s <= SIZE_MIN).sum())} | too-big {int((_s >= SIZE_MAX).sum())} | "
      f"at-cap {int((_s >= SIZE_GROW_CAP).sum())}")
print(f"accepted area: med {np.median(_areas):.0f} px "
      f"({2*np.sqrt(np.median(_areas)/np.pi)*UM_PER_PX:.1f} um equiv. diameter) | "
      f">1.7x med (merge-suspect): {int((_areas > 1.7*np.median(_areas)).sum())}")

_hi, _inroi = final_anatomy > np.percentile(final_anatomy, 99), roi_map > 0
_cl, _un = _hi & _inroi, _hi & ~_inroi
print(f"bright px {int(_hi.sum())} | claimed {int(_cl.sum())} | unclaimed {int(_un.sum())}")
if _cl.sum() and _un.sum():
    _mc, _mu = np.median(corr_bordered[_cl]), np.median(corr_bordered[_un])
    print(f"corr @ bright+claimed   {_mc:.3f}")
    print(f"corr @ bright+unclaimed {_mu:.3f}")
    print("   -> comparable  = cells are being MISSED; parameters still have room."
          if abs(_mc - _mu) < 0.08 else
          "   -> unclaimed much LOWER = those cells are silent in this recording;\n"
          "      no threshold recovers them and this count is the honest yield.")

# READ THIS BEFORE TUNING FURTHER:
#  at-cap > a handful      -> runaway growth, threshold too low
#  med area >> SIZE_MAX/2  -> growth bridging into neuropil
#  huge pile below SIZE_MIN in the sweep -> the size FLOOR is the limiter, not the threshold
#  attempts almost all at the 3x3 seed size (9-19 px) -> nothing is growing: the growth
#     comparison (3x3 sum vs one raw pixel) is starved. Lower the threshold or raise
#     detection SNR; a higher threshold makes it strictly worse.

In [ ]:
# traces at FULL rate from the aligned movie (sum over member pixels; identical
# semantics to the grown sum, order-independent)
if masks:
    traces_raw = np.stack([final_frames[:, m].sum(axis=1) for m in masks]).astype(np.float32)
    roi_npix = np.array([int(m.sum()) for m in masks])
else:
    traces_raw = np.zeros((0, T), np.float32); roi_npix = np.array([], int)
print("traces_raw:", traces_raw.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
mm = np.ma.masked_where(roi_map == 0, roi_map)
axes[0].imshow(mm, cmap="prism", alpha=.45); axes[0].set_title(f"{len(masks)} ROIs"); axes[0].axis("off")
if len(traces_raw):
    tz = np.nan_to_num(zscore(traces_raw, axis=1))   # constant traces -> 0, not NaN
    axes[1].imshow(tz[np.argsort(np.argmax(tz, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(tz, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(tz) + 1])
    axes[1].set(xlabel="time (s)", ylabel="ROI # (viz only: z-scored)")
plt.tight_layout(); plt.show()

## ΔF/F (static bottom-15% F0, team decision) + bleaching check (directive 8)
Static F0 preserves slow arousal-state differences (the science) but **assumes no strong
photobleaching** — under bleach, F0 sits near the late-run floor and inflates early ΔF/F.
So we measure it: a transient-resistant linear fit — the FOV-mean raw F is first reduced
to 1-second MEDIANS (calcium transients barely move a median), then fit with least
squares. If the fitted drift across the run exceeds `BLEACH_WARN_PCT` of the mean, this
run gets flagged and the F0 strategy revisited (per-ROI detrended F0 is the usual
remedy — decide as a team, not silently).

In [ ]:
if len(traces_raw):
    F0 = np.percentile(traces_raw, DFF_PERCENTILE, axis=1, keepdims=True)  # 15th, per ROI
    if (F0 <= 0).any():
        bad = np.where(F0.squeeze() <= 0)[0]
        print(f"WARNING: non-positive F0 for ROI rows {list(bad)} — their dF/F is "
              f"unreliable (denoised/edge traces?). Inspect before using.")
    dff = (traces_raw - F0) / np.maximum(F0, 1e-6)

    # ---- bleaching check (transient-resistant: fit 1-s MEDIANS, not raw meanF) ----
    meanF = traces_raw.mean(axis=0)
    per_sec = max(1, int(round(frame_rate)))
    nblk = len(meanF) // per_sec
    mF_1s = np.median(meanF[:nblk * per_sec].reshape(nblk, per_sec), axis=1)
    t_1s = time_vector[:nblk * per_sec].reshape(nblk, per_sec).mean(axis=1)
    slope, intercept = np.polyfit(t_1s, mF_1s, 1)
    drift_pct = 100.0 * slope * (time_vector[-1] - time_vector[0]) / meanF.mean()
    bleach_flag = abs(drift_pct) > BLEACH_WARN_PCT
    print(f"mean-F drift over run: {drift_pct:+.1f}%  "
          f"{'*** BLEACH FLAG — revisit F0 strategy ***' if bleach_flag else '(ok)'}")

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].imshow(dff[np.argsort(np.argmax(dff, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(dff, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(dff) + 1])
    axes[0].set(ylabel="ROI #", title="dF/F")
    axes[1].plot(time_vector, meanF, lw=.6, label="FOV-mean raw F")
    axes[1].plot(time_vector, intercept + slope * time_vector, "r--",
                 label=f"drift {drift_pct:+.1f}%")
    axes[1].set(xlabel="time (s)", ylabel="mean F"); axes[1].legend()
    plt.tight_layout(); plt.show()

## QC figures — saved, not shown

Every figure lands in the output folder as a PNG and the handle is closed. Over a batch,
`plt.show()` costs render time and figures accumulate until the kernel dies; these four
images are the entire review surface for a run.

In [ ]:
def save_fig(fig, name):
    p = out / f"qc_{name}.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  ", p.name)
    return p

G = globals()
have = lambda *n: all(k in G and G[k] is not None for k in n)
_skipped = []

# --- motion: shift trace + before/after anatomy ---
if have("total_shift_1", "total_shift_2", "original_anatomy"):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(time_vector, total_shift_1, lw=.5, label="pass 1")
    ax[0].plot(time_vector, total_shift_2, lw=.5, label="pass 2")
    ax[0].set(xlabel="time (s)", ylabel="total shift (px)"); ax[0].legend()
    for a, img, t in zip(ax[1:], (original_anatomy, final_anatomy), ("before", "after")):
        a.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
        a.set_title(t); a.axis("off")
    save_fig(fig, "motion")
else:
    _skipped.append("motion")

# --- anatomy + correlation map ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(final_anatomy, cmap="gray", vmin=np.percentile(final_anatomy, 1),
             vmax=np.percentile(final_anatomy, 99))
ax[0].set_title("anatomy")
im = ax[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
ax[1].set_title(f"correlation map (max {corr_bordered.max():.2f})")
for a in ax: a.axis("off")
plt.colorbar(im, ax=ax[1]); save_fig(fig, "corrmap")

# --- correlation distribution with the threshold marked ---
fig, ax = plt.subplots(figsize=(7, 4))
_vv = corr_bordered[corr_bordered > 0]
ax.hist(_vv, bins=120); ax.set_yscale("log")
ax.axvline(CORR_THRESHOLD, color="r", ls="--", label=f"thr {CORR_THRESHOLD}")
ax.set(xlabel="correlation", ylabel="pixels"); ax.legend()
save_fig(fig, "corr_hist")

# --- ROI footprints, on anatomy and on the map that produced them ---
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(final_anatomy, cmap="gray", vmin=np.percentile(final_anatomy, 1),
             vmax=np.percentile(final_anatomy, 99))
ax[0].set_title(f"{len(masks)} ROIs  (thr {CORR_THRESHOLD}, {SIZE_MIN}-{SIZE_MAX} px)")
ax[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("on corr map")
for a in ax:
    a.contour(roi_map > 0, [.5], colors="r", linewidths=.8); a.axis("off")
save_fig(fig, "roi")

# --- traces ---
if have("traces_raw") and len(traces_raw):
    fig, ax = plt.subplots(figsize=(10, 5))
    _z = zscore(traces_raw, axis=1)
    im = ax.imshow(_z[np.argsort(np.argmax(_z, 1))], cmap="afmhot", aspect="auto",
                   vmin=0, vmax=np.percentile(_z, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(_z) + 1])
    ax.set(xlabel="time (s)", ylabel="ROI # (z-scored for display only)")
    plt.colorbar(im, ax=ax); save_fig(fig, "traces_raster")
else:
    _skipped.append("traces_raster")

if have("dff") and len(dff):
    _n = len(dff)
    # one panel per ROI, independently scaled: a dim cell reads as clearly as a bright one
    fig, axes = plt.subplots(_n, 1, figsize=(11, max(2, 1.0 * _n)), sharex=True)
    axes = np.atleast_1d(axes)
    for k, a in enumerate(axes):
        a.plot(time_vector, dff[k], lw=.6, color="k")
        a.set_ylabel(f"{k+1}", rotation=0, ha="right", va="center", fontsize=8)
        a.spines[["top", "right"]].set_visible(False); a.tick_params(labelsize=7)
    axes[-1].set_xlabel("time (s)")
    fig.suptitle(f"dF/F per ROI (n={_n})", y=1.0); plt.tight_layout()
    save_fig(fig, "dff_per_roi")

    # stacked on a SHARED scale: relative amplitudes are honest here
    fig, ax = plt.subplots(figsize=(11, max(4, 0.5 * _n)))
    _step = float(np.nanpercentile(dff, 99.5)) or 1.0
    for k in range(_n):
        ax.plot(time_vector, dff[k] + k * _step, lw=.6)
    ax.set(xlabel="time (s)", ylabel="ROI",
           yticks=[k * _step for k in range(_n)],
           yticklabels=[str(k + 1) for k in range(_n)])
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"dF/F stacked (offset {_step:.2f})")
    plt.tight_layout(); save_fig(fig, "dff_stacked")
else:
    _skipped += ["dff_per_roi", "dff_stacked"]

print("\nskipped (missing vars):", _skipped or "none")
print("->", out)

## Save

One folder per t-series inside `OUT_PARENT`, named for the run. A re-run clears **its own**
outputs first so two parameter sets can never coexist in one folder, and `_COMPLETE` is
written last — its presence means this run finished, which is what makes a batch resumable.

In [ ]:
OUTPUTS = ["traces_raw.npy", "roi_npix.npy", "dff.npy", "F0.npy", "shifts_yx.npy",
           "mean_img.npy", "corr_map.npy", "roi_map.tif", "params.json", "_COMPLETE"]
for _f in OUTPUTS:                     # never mix two runs' files in one folder
    if _f != "shifts_yx.npy":          # keep the shifts written earlier
        (out / _f).unlink(missing_ok=True)

_has = len(traces_raw) > 0
np.save(out / "traces_raw.npy", traces_raw)
np.save(out / "roi_npix.npy",   roi_npix)
np.save(out / "dff.npy", dff if _has else np.empty((0, 0), np.float32))
np.save(out / "F0.npy",  np.atleast_1d(F0.squeeze()) if _has else np.empty(0, np.float32))
np.save(out / "shifts_yx.npy", np.stack([y2, x2]))
np.save(out / "mean_img.npy",  final_anatomy)
np.save(out / "corr_map.npy",  correlation_map)
tiff.imwrite(out / "roi_map.tif", roi_map)

(out / "params.json").write_text(json.dumps({
    "run_name": RUN_NAME, "tseries": DATA_PATH.name,
    "source_path": str(DATA_PATH),
    "movie_file": MOVIE_PATH.name, "metadata_source": META["source"],
    "metadata_notes": META["notes"],
    "pipeline_order": "1d_gauss_filter -> motion_correct -> [pca] -> detect -> extract",
    "frame_period": FRAME_PERIOD, "fps": frame_rate, "n_frames": int(T),
    "um_per_px": UM_PER_PX, "gauss_sec": GAUSS_SEC, "gauss_width_frames": GAUSS_WIDTH,
    "pca_enable": bool(PCA_ENABLE), "pca_rank": PCA_RANK, "detect_bin": int(DETECT_BIN_N),
    "corr_threshold": CORR_THRESHOLD, "soma_diam_um": list(SOMA_DIAM_UM),
    "size_min": SIZE_MIN, "size_max": SIZE_MAX, "size_grow_cap": SIZE_GROW_CAP,
    "border_px": int(border), "dff_percentile": DFF_PERCENTILE,
    "n_rois": int(len(masks)), "max_shift_px": float(max_shift),
}, indent=2))

(out / "_COMPLETE").write_text("")     # LAST: presence == this run finished
_missing = [f for f in OUTPUTS if not (out / f).exists()]
print(f"wrote {len(OUTPUTS)-len(_missing)}/{len(OUTPUTS)} -> {out}"
      + (f"   MISSING: {_missing}" if _missing else ""))
for _f in sorted(out.iterdir()):
    print(f"   {_f.name:24s} {_f.stat().st_size:>12,d} B")

# round-trip check: the only real proof the files are what you think they are
_rt = np.load(out / "traces_raw.npy")
print(f"\ntraces round-trip: {np.array_equal(_rt, traces_raw)} | n_roi {_rt.shape[0]} | "
      f"roi_map labels {int(tiff.imread(out / 'roi_map.tif').max())} (must match)")